Simulate a AT schedule from a trajectory.
- Take in params
- output Simulated AT

Simulate a complete trial. 

In [238]:
import numpy as np
import pandas as pd
from scipy import stats
from pathlib import Path
import shutil

from at_modelling.LotkaVolterraModel import LotkaVolterraModel


Lets call instantiate a lotka volterra model. We can call the paramDic to see what params it has set

In [239]:
#the model is set with default params
lv_model = LotkaVolterraModel()

print(f'Params:', model_example.paramDic)
print(f'StateVars', model_example.stateVars)



Params: {'DMax': 1, 'rS': 0.027, 'rR': 0.027, 'K': 1, 'dD': 1.5, 'dS': 0.0, 'dR': 0.0, 'S0': 0.74, 'R0': 0.01}
StateVars ['S', 'R']


But it wont simulate yet. If you uncomment and run the following block, you should get an error that the model doesn't have an 'initialStateList'

In [240]:
#model_example.Simulate_AT()

We can set our own initial state list, or simply call the Setparams which will get the values from the param dic (R0, S0). Note that a number of arguments can be passed 

In [241]:
#lv_model.initialStateList = [.74, 0.01] #manually pass them
lv_model.SetParams()
lv_model.Simulate_AT(tumourSizeWhenProgressed=1.2)
lv_model.resultsDf





,Time,DrugConcentration,S,R,TumourSize
0,0.000,1.0,0.740000,0.010000,0.750000
1,0.001,1.0,0.739998,0.010000,0.749998
2,0.002,1.0,0.739995,0.010000,0.749995
3,0.003,1.0,0.739993,0.010000,0.749993
4,0.004,1.0,0.739990,0.010000,0.749990
...,...,...,...,...,...
996,455.996,1.0,0.263583,0.636842,0.900425
997,455.997,1.0,0.263583,0.636843,0.900426
998,455.998,1.0,0.263583,0.636845,0.900428
999,455.999,1.0,0.263582,0.636847,0.900429


We see that We can the print the results. The default time resolution is very high. We can clean the df to longer intervals and also be sure to remove post progression obs

In [242]:
lv_model.CleanData()

lv_model.resultsDf

,Time,S,R,TumourSize,DrugConcentration
0,0.0,0.986667,0.013333,1.000000,1.0
1,1.0,0.983326,0.013424,0.996750,1.0
2,2.0,0.979965,0.013516,0.993481,1.0
3,3.0,0.976582,0.013610,0.990192,1.0
4,4.0,0.973175,0.013706,0.986881,1.0
...,...,...,...,...,...
451,451.0,0.353895,0.837404,1.191299,1.0
452,452.0,0.353390,0.839800,1.193189,1.0
453,453.0,0.352892,0.842170,1.195062,1.0
454,454.0,0.352402,0.844516,1.196917,1.0


## part 2
We can use the model simulate an entire trial. We can the use this data to test out inferential pipeline such as a bayesian fitting routine. We can add some makeshift, but the proper way to do it would be flexible model code. 

In [243]:
priorDic = {
    "cost":     stats.beta(5, 10),
    "turnover": stats.beta(2, 2),
    "n0":       stats.beta(2, 2),
    "rFrac":    stats.beta(2, 30),
    "K":        stats.lognorm(s=0.5, scale=2),       
    "rS":       stats.lognorm(s=0.25, scale=0.019),
    "DD":       stats.uniform(loc=0.5, scale=1.5),   
    'sigma':    stats.lognorm(s=1, scale=np.exp(-2)),  
}

def draw_params(priorDic : dict):
    """Contstruct a dictionary of sampled values from prior dict"""
    return {paramName: dist.rvs() for paramName, dist in priorDic.items()}

draw_params(priorDic)


{'cost': np.float64(0.28223704092937996),
 'turnover': np.float64(0.45803513206598895),
 'n0': np.float64(0.46414734087238607),
 'rFrac': np.float64(0.05668086508554268),
 'K': np.float64(4.940397256473547),
 'rS': np.float64(0.022911159748216003),
 'DD': np.float64(1.9421323061081175),
 'sigma': np.float64(0.08432954240183531)}

In [ ]:
def random_params(paramRange):
    """Selects parameters randomly from output range"""
    output = {}
    for param in paramRange():
        if isinstance(paramRange[key], list):
            assert len(paramRange[key])==2, "Lists have upper and lower bound only"
            output[key] = np.random.uniform(*paramRange[key])
        else:
            output[key] = paramRange[key]
    return output

def complete_param_dict(cost, turnover, n0, rFrac, rS, K, DD, sigma):
    """Pass unpacked dict to generate param dict in
    form required by model class.
    N.B rS is a defualt value from literatue"""
    return {"Cost": cost, "Turnover": turnover,
                "n0": n0, "R0": rFrac * n0, "S0": (n0 - rFrac*n0),
                "rS": rS, "rR": rS * (1-cost),
                "dS": rS * turnover, "dR": rS * turnover,
                'K': K, 'dD': DD}

def test_progression_at(model, t_end):
    """Selects for patients that reach progression (20% growth
    from initial size) within t_end days, under rule of thumb 
    AT treatment.
    """
    output = max(model.resultsDf['Time']) + day_interval < t_end
    return int(output)

# Set allowed paramter ranges (use scalars for set values):
paramRange = {"n0": [0.1, 0.9], 
              "r0": [0.001, 0.5],
              "cost": [0.2, 0.5], 
              "turnover": [0.5, 0.8],
              "K":[1, 5], 
              "DD": [.5, 2],
             "rS": [0.05, .5]}

# Generate random patients
n_patients = 5
check_progression = True
t_end = 5000  # Ensure progression occurs in this period

#steady state check
def progress_on_ct(paramDic):
    """Return true if the params would even progress on continous therapy using the derivsation of Kit Gallagher'
    s DRL paper."""
    progressCTflag = paramDic['Turnover'] < (1 - 1.2 * (paramDic['S0']+paramDic['R0'])/paramDic['K']) * (1 - paramDic['Cost'])
    return(progressCTflag)

#setup path
out_dir = Path('../synth_data')
shutil.rmtree(out_dir, ignore_errors=True) #delete if exists
out_dir.mkdir(parents=True)\

for i in range(n_patients):
    #run loop until viable param combination is found
    while True:
        priorDrawDic = draw_params(priorDic)
        paramDic = complete_param_dict(**priorDrawDic)
        if not progress_on_ct(paramDic=paramDic):
            print("Analytically doesnt progress CT")
            continue
        model = LotkaVolterraModel(method='RK45', dt=1)
        model.SetParams(**paramDic)
        model.Simulate_AT(atThreshold=0.5, intervalLength=1, t_end=t_end)
        if not np.isnan(model.ttp) or not check_progression:
            print(f"Check passed or not checking progression on AT! Saving Patient {i} out of {n_patients}")
            break
        print("Check failed")
    #break out while loop and store found param combination
    model.CleanData()
    resultsDf = model.resultsDf
    resultsDf.to_csv(f'../synth_data/patient_{i}.csv')
    priorDrawDic["PatientId"] = i
    if i == 0:
        df = pd.DataFrame(priorDrawDic, index=[0])
    else:
        df = pd.concat([df, pd.DataFrame(priorDrawDic, index=[i])])
df.to_csv(f"../synth_data/randSynthetic{n_patients}Patients.csv")



Check passed or not checking progression on AT! Saving Patient 0 out of 5
Check passed or not checking progression on AT! Saving Patient 1 out of 5
Check passed or not checking progression on AT! Saving Patient 2 out of 5
Analytically doesnt progress CT
Analytically doesnt progress CT
Analytically doesnt progress CT
Check passed or not checking progression on AT! Saving Patient 3 out of 5
Check passed or not checking progression on AT! Saving Patient 4 out of 5
